# Clase 11 · Caso Panorama de la vivienda, parte 2 — práctica guiada

El profesor usó Melbourne (`Price`). Aquí repetimos **cada paso** con `data/raw/viviendas_lima.csv` (objetivo `precio_usd`).

Recorrido:
1. Separar X e y (y sin nulos).
2. Imputar X con `KNNImputer`: **mismos tamaños**.
3. Detectar outliers **por fila** y filtrar X e y con la **misma máscara**.
4. Dividir 80/20 y medir el MAE en varios escenarios.
5. El debate: ¿imputar **antes** o **después** de dividir? Lo medimos.
6. `PowerTransformer` (Box-Cox / Yeo-Johnson).

> Si `data/raw/` está vacío, ejecuta primero `python data/generar_datos.py` desde la raíz del repo.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors

warnings.filterwarnings("ignore")

# Busca la carpeta data/raw subiendo desde donde esté abierto el notebook
raiz = Path.cwd()
while not (raiz / "data" / "raw").exists() and raiz != raiz.parent:
    raiz = raiz.parent
df = pd.read_csv(raiz / "data" / "raw" / "viviendas_lima.csv")
df.shape

## 1. Separar X e y
*"Primero quiero mantener separado mi X de mi Y."* Solo las **numéricas** (las categóricas, en la próxima clase). Quitamos también `id` (📌 Extra: no describe la casa).

Antes de nada, comprobamos que el objetivo **no tenga nulos**. Si los tuviera, *"ya es otro dolor de cabeza"*.

In [ ]:
print("Nulos en precio_usd:", df["precio_usd"].isna().sum())

X = df.select_dtypes(include="number").drop(columns=["precio_usd", "id"])
y = df["precio_usd"]
print(X.shape, y.shape)
X.isna().sum()

Los nulos están **solo en los predictores**. Fíjate en `piso`: sus nulos son todos **casas** (una casa no tiene piso). Eso es un nulo *estructural*, como viste en el módulo 1.

In [ ]:
df.loc[df["piso"].isna(), "tipo_inmueble"].value_counts()

## 2. Imputar con `KNNImputer`
`fit_transform` devuelve un **array** de numpy. Lo volvemos DataFrame con las columnas **y el índice** de X: si pierdes el índice, después no podrás alinear X con y.

Regla del profesor: *"estos tamaños deben coincidir sí o sí"*.

In [ ]:
X_imp = pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(X), columns=X.columns, index=X.index)

print("X_imp:", X_imp.shape, "| y:", y.shape)
print("Nulos que quedan:", X_imp.isna().sum().sum())
assert len(X_imp) == len(y), "¡Los tamaños deben coincidir!"


### ¿Por qué no la mediana? (pregunta de Arturo)
Con la mediana, todos los nulos caen en **el mismo valor** y aparece un **pico**. KNN, en cambio, rellena según las casas parecidas. Compara los valores que recibió `distancia_centro_km` con cada método:

In [ ]:
faltaban = X["distancia_centro_km"].isna()
por_mediana = X["distancia_centro_km"].fillna(X["distancia_centro_km"].median())[faltaban]
por_knn = X_imp.loc[faltaban, "distancia_centro_km"]

print("Valores distintos imputados -> mediana:", por_mediana.nunique(), "| KNN:", por_knn.nunique())
pd.DataFrame({"mediana": por_mediana.describe(), "knn": por_knn.describe()}).round(2)

## 3. Quitar outliers **por fila**
En clase: `KNN(contamination=0.1).fit(X_imp)` de **PyOD** y luego `labels_` (0 = inlier, 1 = outlier). PyOD no está instalado aquí, pero su KNN es simple: **la distancia de cada fila a su k-ésimo vecino**. El 10 % de filas con mayor distancia se marca como outlier. Lo reproducimos con `NearestNeighbors`:

In [ ]:
k = 5
nn = NearestNeighbors(n_neighbors=k + 1).fit(X_imp)       # +1 porque cada punto es su propio vecino
distancias, _ = nn.kneighbors(X_imp)
puntaje = distancias[:, -1]                                # distancia al k-ésimo vecino (method="largest")

umbral = np.percentile(puntaje, 100 * (1 - 0.1))           # contamination = 0.1
labels_ = (puntaje > umbral).astype(int)                   # convención PyOD: 0 inlier, 1 outlier
pd.Series(labels_).value_counts()

**Por fila, no por variable** (pregunta de Samuel): cada casa es **un punto** en un espacio de 7 dimensiones. Veamos las 5 casas más "lejanas" de sus vecinas:

In [ ]:
X_imp.assign(puntaje=puntaje, precio=y).sort_values("puntaje", ascending=False).head()

Ahora con scikit-learn. **Ojo a la convención:** `fit_predict` devuelve **1 = inlier, -1 = outlier** (PyOD usa 0/1). Lo más claro es convertirlo a una **máscara booleana** (`True` = me quedo con la fila):

In [ ]:
pred_if = IsolationForest(contamination=0.1, random_state=0).fit_predict(X_imp)
pred_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1).fit_predict(X_imp)

mascara_if = pd.Series(pred_if == 1, index=X_imp.index, name="inlier")
mascara_lof = pd.Series(pred_lof == 1, index=X_imp.index, name="inlier")
mascara_knn = pd.Series(labels_ == 0, index=X_imp.index, name="inlier")

print("Inliers -> IsolationForest:", mascara_if.sum(), "| LOF:", mascara_lof.sum(), "| KNN (PyOD manual):", mascara_knn.sum())
print("¿Coinciden IF y LOF en cuáles quitar?", int((~mascara_if & ~mascara_lof).sum()), "outliers en común")

**Filtrar X e y con la MISMA máscara.** *"En ningún momento he tocado a y; aquí simplemente es un filtrado, porque los índices coinciden."*

In [ ]:
X_r = X_imp.loc[mascara_if]
y_r = y.loc[mascara_if]

print("Antes:", X_imp.shape, "-> después:", X_r.shape, "| y_r:", y_r.shape)
assert X_r.index.equals(y_r.index), "X_r e y_r deben estar alineados"
type(X_r)   # sigue siendo DataFrame porque filtramos con .loc (el profe tuvo que reconvertir un array)

## 4. Dividir 80/20 y medir el MAE
Usamos el mismo modelo que `score_dataset` de la clase 10: RandomForest con 50 árboles, `max_depth=6` y `random_state=0`.

In [ ]:
def score_dataset(X_train, X_test, y_train, y_test):
    modelo = RandomForestRegressor(n_estimators=50, max_depth=6, random_state=0)
    modelo.fit(X_train, y_train)
    return mean_absolute_error(y_test, modelo.predict(X_test))


def escenario(Xs, ys):
    return round(score_dataset(*train_test_split(Xs, ys, test_size=0.2, random_state=0)), 2)


X_med = X.fillna(X.median())
resultados = {
    "mediana, sin quitar outliers": escenario(X_med, y),
    "KNNImputer, sin quitar outliers": escenario(X_imp, y),
    "KNNImputer + IsolationForest": escenario(X_imp[mascara_if], y[mascara_if]),
    "KNNImputer + LOF": escenario(X_imp[mascara_lof], y[mascara_lof]),
    "KNNImputer + KNN (PyOD manual)": escenario(X_imp[mascara_knn], y[mascara_knn]),
}
pd.Series(resultados, name="MAE (USD)").sort_values()

📌 **Lectura crítica.** Al quitar outliers **antes** de dividir, el test **también** pierde sus casas raras. Parte de la mejora viene de que el examen se volvió más fácil. Una comparación más justa evalúa todos los escenarios **sobre el mismo test**: dividimos primero, limpiamos solo train y medimos sobre el test completo.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)

imp = KNNImputer(n_neighbors=5).fit(Xtr)                                  # fit SOLO en train
Xtr_imp = pd.DataFrame(imp.transform(Xtr), columns=X.columns, index=Xtr.index)
Xte_imp = pd.DataFrame(imp.transform(Xte), columns=X.columns, index=Xte.index)

m_tr = IsolationForest(contamination=0.1, random_state=0).fit_predict(Xtr_imp) == 1   # outliers solo de train
print("Sin quitar outliers      :", round(score_dataset(Xtr_imp, Xte_imp, ytr, yte), 2))
print("Quitando outliers (train):", round(score_dataset(Xtr_imp[m_tr], Xte_imp, ytr[m_tr], yte), 2))

## 5. El debate: ¿imputar antes o después de dividir?
El profesor prefiere **antes** (integridad del test, evitar fuga, control del proceso). La práctica estándar de scikit-learn es **después** (`fit` solo en train). Mejor que opinar, **medimos**: mismo split, mismo modelo, y solo cambia dónde se hace el `fit` del imputador.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)

imp_antes = SimpleImputer(strategy="median").fit(X)       # ve TODO X (incluye test)
imp_despues = SimpleImputer(strategy="median").fit(Xtr)   # ve solo train

medianas = pd.DataFrame({
    "antes (todo X)": imp_antes.statistics_,
    "después (solo train)": imp_despues.statistics_,
}, index=X.columns)
display(medianas)

mae_antes = score_dataset(imp_antes.transform(Xtr), imp_antes.transform(Xte), ytr, yte)
mae_despues = score_dataset(imp_despues.transform(Xtr), imp_despues.transform(Xte), ytr, yte)
print(f"MAE antes  : {mae_antes:,.2f}")
print(f"MAE después: {mae_despues:,.2f}")
print(f"Diferencia : {mae_despues - mae_antes:,.2f}")

Las medianas "antes" **conocen el test**. La diferencia de MAE es pequeña: con suficientes datos el efecto es chico, como dijo el profesor. Pero la versión "después" es la que imita lo que pasará en producción, donde la casa nueva no existe cuando entrenas.

### ✏️ Tu turno 1
Repite la comparación con **pocos datos**: usa solo las primeras 200 filas (`df.head(200)`). ¿La diferencia entre "antes" y "después" crece? Esto pone a prueba la frase del profesor: *"cuando hay poca data puede ser que sí veamos un efecto"*.

In [ ]:
# ✏️ Tu código aquí


### ✏️ Tu turno 2
La tarea del profesor: *"prueben más escenarios"*. Prueba `contamination` = 0.05 y 0.2 con IsolationForest, y `n_neighbors` = 3 y 10 en el `KNNImputer`. Usa la comparación **justa** (mismo test) del punto 4. ¿Cuál combinación da el menor MAE?

In [ ]:
# ✏️ Tu código aquí


## 6. `PowerTransformer` (Box-Cox / Yeo-Johnson)
Se usa cuando lo que viene después **necesita normalidad** (por ejemplo, PCA). **Box-Cox** exige valores **positivos**. **Yeo-Johnson** (el nombre que el profesor no recordaba; es el método por defecto) acepta ceros y negativos.

Como **aprende** un λ por columna, va **después de dividir**: `fit` en train y `transform` en test.

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method="yeo-johnson").fit(Xtr_imp[["area_m2", "distancia_centro_km"]])
print("lambdas aprendidos:", pt.lambdas_.round(3))
transformado = pd.DataFrame(pt.transform(Xte_imp[["area_m2", "distancia_centro_km"]]), columns=["area_m2", "distancia_centro_km"])
pd.DataFrame({"skew original (test)": Xte_imp[["area_m2", "distancia_centro_km"]].skew(),
              "skew transformado (test)": transformado.skew()}).round(2)

### ✏️ Tu turno 3
Prueba `method="box-cox"` sobre `antiguedad_anios`. ¿Qué error sale y por qué? (Pista: ¿hay casas con antigüedad 0?)

In [ ]:
# ✏️ Tu código aquí


## Resumen
- La imputación **no** cambia tamaños. Quitar outliers **sí** quita filas, y hay que filtrar X **e** y con la misma máscara.
- Los detectores trabajan **por fila** (cada casa es un punto en R^n).
- Convenciones: PyOD usa 0/1, scikit-learn usa 1/-1, y aquí trabajamos con `True`/`False`.
- Escalamiento, PCA y `PowerTransformer` van **siempre** después de dividir. La imputación y los outliers son un debate: mídelo y, en caso de duda, ajusta solo con train.

Ahora ve a `ejercicio.py`.